In [1]:
import os
import torch
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A

class AugmentDataset(Dataset):
    def __init__(self, csv_path, image_dir, mask_dir, split='train', augment=True):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.augment = augment

    def randomHorizontalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 1)
            mask = cv2.flip(mask, 1)
        return image, mask

    def randomVerticalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 0)
            mask = cv2.flip(mask, 0)
        return image, mask

    def randomRotate90(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = np.rot90(image).copy()
            mask = np.rot90(mask).copy()
        return image, mask

    def randomHueSaturationValue(self, image, hue_shift_limit=(-30, 30),
                                  sat_shift_limit=(-5, 5),
                                  val_shift_limit=(-15, 15), u=0.5):
        if np.random.rand() < u:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
            h, s, v = cv2.split(image)
            h = h.astype(np.float32)
            s = s.astype(np.float32)
            v = v.astype(np.float32)


            h += np.random.randint(hue_shift_limit[0], hue_shift_limit[1] + 1)
            s += np.random.uniform(sat_shift_limit[0], sat_shift_limit[1])
            v += np.random.uniform(val_shift_limit[0], val_shift_limit[1])

            h = np.clip(h, 0, 179).astype(np.uint8)
            s = np.clip(s, 0, 255).astype(np.uint8)
            v = np.clip(v, 0, 255).astype(np.uint8)

            image = cv2.merge((h, s, v))
            image = cv2.cvtColor(image, cv2.COLOR_HSV2RGB)
        return image

    def randomShiftScaleRotate(self, image, mask,
                               shift_limit=(-0.1, 0.1),
                               scale_limit=(-0.1, 0.1),
                               rotate_limit=(0, 0),
                               aspect_limit=(-0.1, 0.1),
                               borderMode=cv2.BORDER_CONSTANT, u=0.5):
        if np.random.rand() < u:
            height, width = image.shape[:2]
            angle = np.random.uniform(*rotate_limit)
            scale = np.random.uniform(1 + scale_limit[0], 1 + scale_limit[1])
            aspect = np.random.uniform(1 + aspect_limit[0], 1 + aspect_limit[1])
            sx = scale * aspect / (aspect ** 0.5)
            sy = scale / (aspect ** 0.5)
            dx = int(np.random.uniform(*shift_limit) * width)
            dy = int(np.random.uniform(*shift_limit) * height)

            cc = np.cos(np.radians(angle)) * sx
            ss = np.sin(np.radians(angle)) * sy
            rotation = np.array([[cc, -ss], [ss, cc]])

            box = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)
            box -= np.array([width / 2, height / 2], dtype=np.float32)
            box = np.dot(box, rotation.T) + np.array([width / 2 + dx, height / 2 + dy], dtype=np.float32)

            mat = cv2.getPerspectiveTransform(box.astype(np.float32), np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32))
            image = cv2.warpPerspective(image, mat, (width, height), flags=cv2.INTER_LINEAR, borderMode=borderMode, borderValue=(0, 0, 0))
            mask = cv2.warpPerspective(mask, mat, (width, height), flags=cv2.INTER_NEAREST, borderMode=borderMode, borderValue=(0,))

        return image, mask
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        mask_path = os.path.join(self.mask_dir, row['maskname'])

        # Read image & mask
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Augment
        if self.augment:
            image = self.randomHueSaturationValue(image)
            image, mask = self.randomShiftScaleRotate(image, mask)
            image, mask = self.randomHorizontalFlip(image, mask)
            image, mask = self.randomVerticalFlip(image, mask)
            image, mask = self.randomRotate90(image, mask)

        # Normalize image
        image = image.astype(np.float32) / 255.0
        image = image * 3.2 - 1.6
        image = np.transpose(image, (2, 0, 1))  # HWC → CHW

        # Normalize mask
        mask = mask.astype(np.float32) / 255.0
        mask = np.expand_dims(mask, axis=0)  # (1, H, W)
        mask = (mask > 0.5).astype(np.float32)

        return torch.tensor(image), torch.tensor(mask)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class DBlock(nn.Module):
    def __init__(self, channels):
        super(DBlock, self).__init__()
        self.dilate1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, dilation=1)
        self.dilate2 = nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2)
        self.dilate3 = nn.Conv2d(channels, channels, kernel_size=3, padding=4, dilation=4)
        self.dilate4 = nn.Conv2d(channels, channels, kernel_size=3, padding=8, dilation=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        d1 = self.relu(self.dilate1(x))
        d2 = self.relu(self.dilate2(d1))
        d3 = self.relu(self.dilate3(d2))
        d4 = self.relu(self.dilate4(d3))

        return x + d1 + d2 + d3 + d4
    
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        mid_channels = in_channels // 4

        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.relu1 = nn.ReLU(inplace=True)

        # Simulate ExpandDims + Conv3DTranspose + Squeeze
        self.deconv3d = nn.ConvTranspose3d(
            mid_channels, mid_channels,
            kernel_size=(1, 3, 3),
            stride=(1, 2, 2),
            padding=(0, 1, 1),
            output_padding=(0, 1, 1)
        )

        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.relu2 = nn.ReLU(inplace=True)

        self.conv3 = nn.Conv2d(mid_channels, out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        # Expand dims
        x = x.unsqueeze(2)  # (B, C, 1, H, W)
        x = self.deconv3d(x)
        x = x.squeeze(2)    # (B, C, H*2, W*2)

        x = self.bn2(x)
        x = self.relu2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)

        return x

class DLinkNet34(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(DLinkNet34, self).__init__()
        filters = [64, 128, 256, 512]

        resnet = models.resnet34(pretrained=pretrained)

        # Encoder
        self.firstconv = resnet.conv1
        self.firstbn = resnet.bn1
        self.firstrelu = resnet.relu
        self.firstmaxpool = resnet.maxpool

        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

        # Center
        self.dblock = DBlock(512)

        # Decoder
        self.decoder4 = DecoderBlock(filters[3], filters[2])
        self.decoder3 = DecoderBlock(filters[2], filters[1])
        self.decoder2 = DecoderBlock(filters[1], filters[0])
        self.decoder1 = DecoderBlock(filters[0], filters[0])

        # Final conv
        self.finaldeconv1 = nn.ConvTranspose2d(filters[0], 32, kernel_size=4, stride=2, padding=1)
        self.finalrelu1 = nn.ReLU(inplace=True)
        self.finalconv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.finalrelu2 = nn.ReLU(inplace=True)
        self.finalconv3 = nn.Conv2d(32, num_classes, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        x = self.firstconv(x)
        x = self.firstbn(x)
        x = self.firstrelu(x)
        x = self.firstmaxpool(x)

        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        # Center
        center = self.dblock(e4)

        # Decoder
        d4 = self.decoder4(center) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)

        out = self.finaldeconv1(d1)
        out = self.finalrelu1(out)
        out = self.finalconv2(out)
        out = self.finalrelu2(out)
        out = self.finalconv3(out)

        return out


In [4]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def get_optimizer(model, lr=5e-4, weight_decay=1e-4):
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def get_scheduler(optimizer, T_0=10, T_mult=2, eta_min=1e-6):
    return CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=T_0, 
    T_mult=T_mult, 
    eta_min=eta_min
)

In [5]:
def bce_dice_loss(y_pred_logits, y_true, smooth=1e-5):
    """
    y_pred_logits: Raw model outputs (NO Sigmoid applied in the architecture)
    y_true: Ground truth binary masks
    """
    # 1. Numerically stable BCE using Logits
    bce = F.binary_cross_entropy_with_logits(y_pred_logits, y_true)
    
    # 2. Apply Sigmoid safely inside the loss for Dice calculation
    y_pred_probs = torch.sigmoid(y_pred_logits)
    
    # 3. Flatten the ENTIRE batch into a single 1D vector for maximum GPU speed
    y_pred_flat = y_pred_probs.view(-1)
    y_true_flat = y_true.view(-1)
    
    # 4. Global Dice calculation
    intersection = (y_pred_flat * y_true_flat).sum()
    dice = (2. * intersection + smooth) / (y_pred_flat.sum() + y_true_flat.sum() + smooth)
    
    dice_loss = 1.0 - dice
    
    # Combined loss
    return bce + dice_loss

In [6]:
import torch
import os

def save_model(model, path, epoch, score):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state = {
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'score': score
    }
    torch.save(state, path)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
# === Cấu hình ===
batch_size = 16
num_epochs = 100
lr = 5e-4
log_dir = '/kaggle/working/logs'
model_path = '/kaggle/working/best_model.pth'
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'

In [9]:
# === Dataloader ===
train_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='train', augment=True)
val_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='val', augment=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4,pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,pin_memory=True)

In [10]:
# Previous-run best model
input_model_path = '/kaggle/input/models/vuongtran11233/dlinknet-old1/pytorch/default/1/best_model.pth'

In [11]:
from collections import OrderedDict

model = DLinkNet34(num_classes=1)
checkpoint = torch.load(input_model_path, map_location=device)

state_dict = checkpoint['model_state_dict']
clean_state_dict = OrderedDict()

for key, value in state_dict.items():
    if key.startswith('module.'):
        clean_name = key[7:]  # 'module.' is 7 characters long
        clean_state_dict[clean_name] = value
    else:
        clean_state_dict[key] = value
        
model.load_state_dict(clean_state_dict)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 168MB/s]


<All keys matched successfully>

In [12]:
# === Optimizer ===
#model = DLinkNet34(num_classes=1)
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs for parallel training!")
    model = nn.DataParallel(model)
model.to(device)
optimizer = get_optimizer(model, lr)
scheduler = get_scheduler(optimizer)
EarlyStopPatience = 15

🚀 Using 2 GPUs for parallel training!


In [13]:
import gc

In [14]:
# === Training Loop ===
best_loss = 999.0
for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc='Training'):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # === Validation ===
    model.eval()
    val_loss = 0
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc='Validation'):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # === Scheduler + EarlyStopping ===
    scheduler.step(val_loss)
    
    if val_loss < best_loss:
        best_loss = val_loss
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        save_model(model, model_path, epoch, best_loss)
        patience_counter = 0  # Reset counter since we improved
        print("Validation loss improved!")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{EarlyStopPatience}")

    # Trigger early stopping
    if patience_counter >= EarlyStopPatience:
        print("Early stopping triggered. Stopping training!")
        break

    del outputs, loss
    gc.collect()
    torch.cuda.empty_cache()

print("Training completed!")



Epoch [1/100]


Validation: 100%|██████████| 70/70 [00:48<00:00,  1.45it/s]


Train Loss: 0.3164 | Val Loss: 0.3347
Validation loss improved!

Epoch [2/100]


Validation: 100%|██████████| 70/70 [00:48<00:00,  1.46it/s]


Train Loss: 0.3149 | Val Loss: 0.3240
Validation loss improved!

Epoch [3/100]


...

Validation: 100%|██████████| 70/70 [00:50<00:00,  1.37it/s]


Train Loss: 0.2885 | Val Loss: 0.3087
Validation loss improved!

Epoch [41/100]


Validation: 100%|██████████| 70/70 [00:50<00:00,  1.38it/s]


Train Loss: 0.2900 | Val Loss: 0.3121
No improvement. Patience: 1/15

Epoch [42/100]


Training:  83%|████████▎ | 269/324 [08:48<02:18,  2.53s/it]